# DUMPLINGs Colab Assistant Journal Control Panel

A lightweight Colab workflow for rebuilding factual indices and running the assistant journals on copied `runs/` artifacts.

This notebook is intentionally separate from the training notebook. It stages only the light analysis workspace into `/content`.

In [ ]:
from pathlib import Path
import subprocess
import os

DRIVE_REPO_ROOT = "/content/drive/MyDrive/DUMPLINGs"
COLAB_WORKSPACE = "/content/DUMPLINGs_analysis"

RUN_PREFIXES = [
    "DUMPLING_A1_esm_only",
    "DUMPLING_A1_dimenet_only",
    "DUMPLING_A1_full",
    "DUMPLING_A1_dimenet_esm"
]

ASSISTANT_MODE = "live"  # "dry-run" or "live"
ASSISTANT_MODEL = "qwen2.5:14b"
ASSISTANT_LIMIT = 0       # 0 means all copied runs
FORCE_REFRESH = False
OLLAMA_TIMEOUT_SEC = 1800

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
SRC = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs"
DST = "/content/DUMPLINGs"

LOCAL_RUNS = f"{DST}/runs"
DRIVE_RUNS = f"{SRC}/runs"

os.chdir(SRC)

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("Drive source repo aligned to origin/main")
print("Drive repo:", SRC)
print("Working copy will be staged into:", DST)
print("Git commit:", commit)

Drive source repo aligned to origin/main
Drive repo: /content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs
Working copy will be staged into: /content/DUMPLINGs
Git commit: a46b2129a3cf962a8eadee467e348bdbfabcab3c


In [18]:
import subprocess
import sys

cmd = [
    sys.executable,
    f"{SRC}/scripts/colab_stage_analysis_workspace.py",
    "--src", SRC,
    "--dst", DST,
]
for prefix in RUN_PREFIXES:
    cmd.extend(["--include-run-prefix", prefix])

subprocess.run(cmd, check=True)
print(f"Staged analysis workspace into {DST}")

Staged analysis workspace into /content/DUMPLINGs


In [19]:
%cd /content/DUMPLINGs
!python3 scripts/rebuild_experiment_index.py --runs-dir runs
!ls

/content/DUMPLINGs
Rebuilt registry: /content/DUMPLINGs/runs/experiment_registry.csv
Rebuilt journal:  /content/DUMPLINGs/runs/experiment_journal.md
Rebuilt series:   /content/DUMPLINGs/runs/experiment_series_journal.md
Indexed runs:     40
assistant  configs  README.md  requirements.txt  runs  scripts


In [20]:
! ls runs
!echo "SRC runs:"
!ls -la "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs"

!echo "\nALT runs:"
!ls -la "/content/drive/MyDrive/DUMPLINGs/runs"


DUMPLING_A1_dimenet_esm_20260512_054215_911987
DUMPLING_A1_dimenet_esm_20260512_063855_876071
DUMPLING_A1_dimenet_esm_20260512_072111_231421
DUMPLING_A1_dimenet_esm_20260512_080346_079659
DUMPLING_A1_dimenet_esm_20260512_085336_670324
DUMPLING_A1_dimenet_esm_20260512_095207_477182
DUMPLING_A1_dimenet_esm_20260512_105254_771496
DUMPLING_A1_dimenet_esm_20260512_115948_179002
DUMPLING_A1_dimenet_esm_20260512_131434_679947
DUMPLING_A1_dimenet_esm_20260512_143237_511628
DUMPLING_A1_dimenet_only_20260511_145527_990905
DUMPLING_A1_dimenet_only_20260511_163554_991527
DUMPLING_A1_dimenet_only_20260511_183303_002437
DUMPLING_A1_dimenet_only_20260511_201237_121235
DUMPLING_A1_dimenet_only_20260511_214404_634398
DUMPLING_A1_dimenet_only_20260511_231406_720927
DUMPLING_A1_dimenet_only_20260512_003120_495078
DUMPLING_A1_dimenet_only_20260512_014412_124637
DUMPLING_A1_dimenet_only_20260512_031925_030764
DUMPLING_A1_dimenet_only_20260512_041454_408193
DUMPLING_A1_esm_only_20260511_160803_148522
DUMPLI

In [31]:
!apt-get update
!apt-get install -y zstd

import subprocess

cmd = "curl -fsSL https://ollama.com/install.sh | sh"
result = subprocess.run(cmd, shell=True, text=True, capture_output=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,616 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]


In [41]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path

if ASSISTANT_MODE == "live":
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(
        "OLLAMA_HOST=127.0.0.1:11434 ollama serve > /tmp/ollama.log 2>&1",
        shell=True,
    )
    health_url = "http://127.0.0.1:11434/api/tags"
    for _ in range(60):
        try:
            with urllib.request.urlopen(health_url, timeout=3):
                break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama did not become ready in time")

    subprocess.run(["ollama", "pull", ASSISTANT_MODEL], check=True)
    env_path = Path("assistant/.env")
    env_path.write_text(
        "\n".join([
            "ASSISTANT_LLM_PROVIDER=ollama",
            f"ASSISTANT_LLM_MODEL={ASSISTANT_MODEL}",
            f"ASSISTANT_LLM_TIMEOUT_SEC={OLLAMA_TIMEOUT_SEC}",
            "ASSISTANT_LLM_TEMPERATURE=0.2",
        ]) + "\n",
        encoding="utf-8",
    )
    print(env_path.read_text(encoding="utf-8"))
else:
    print("Dry-run mode selected; skipping Ollama install and model pull.")

ASSISTANT_LLM_PROVIDER=ollama
ASSISTANT_LLM_MODEL=qwen2.5:14b
ASSISTANT_LLM_TIMEOUT_SEC=1800
ASSISTANT_LLM_TEMPERATURE=0.2



In [42]:
import subprocess

cmd = ["bash", "assistant/run_llm_journal.sh", f"--{ASSISTANT_MODE}"]
if ASSISTANT_LIMIT > 0:
    cmd.extend(["--limit", str(ASSISTANT_LIMIT)])
if FORCE_REFRESH:
    cmd.append("--force-refresh")

subprocess.run(cmd, check=True)

CompletedProcess(args=['bash', 'assistant/run_llm_journal.sh', '--live'], returncode=0)

In [43]:
from pathlib import Path

for rel_path in [
    "runs/experiment_journal_llm.md",
    "runs/experiment_series_journal_llm.md",
]:
    path = Path(rel_path)
    print(f"\n===== {rel_path} =====\n")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:12000])
    else:
        print("missing")


===== runs/experiment_journal_llm.md =====

# Experiment Journal LLM

This file is the manual LLM-backed experiment journal. Entry structure mirrors the factual journal, but assistant notes are produced by the separate assistant layer.

## 2026-05-12T06:38:46 | DUMPLING_A1_dimenet_esm_20260512_054215_911987

- status: `success` | model: `A1` | env: `cluster` | seed: `42` | duration_sec: `3384.6`
- location: `runs/DUMPLING_A1_dimenet_esm_20260512_054215_911987` on `corsa`
- final metrics: RMSE=`1.363833616589381`, Pearson_R=`0.7833619876876898`, CI=`0.7902327027479492`
- artifacts: [folder](DUMPLING_A1_dimenet_esm_20260512_054215_911987/) | [config](DUMPLING_A1_dimenet_esm_20260512_054215_911987/config.json) | [summary](DUMPLING_A1_dimenet_esm_20260512_054215_911987/assistant_summary.md) | [history](DUMPLING_A1_dimenet_esm_20260512_054215_911987/history.json) | [test](DUMPLING_A1_dimenet_esm_20260512_054215_911987/test_results.json) | [report](DUMPLING_A1_dimenet_esm_20260512_054215_91

## Notes

- This notebook is happy with copied `runs/` folders; it does not need the training stack.
- If you want to preserve the generated LLM journals back to Drive, copy the files from `/content/DUMPLINGs_analysis/runs/` into your Drive repo after inspection.
- If Colab GPU memory is tight, try a smaller Ollama model tag before concluding the workflow is bad.